# SQL & SQLite ? Complete Guide

## Architecture Diagram
[Python App] -> [sqlite3.connect()] -> [Connection] -> [cursor()] -> [Cursor]
-> [cursor.execute(SQL)] -> [SQLite Engine reads/writes example.db file]
-> [cursor.fetchall()/fetchone()] -> [Python tuples] -> [connection.commit() saves to disk]

## Deep Architecture Notes
- Step 1: `sqlite3.connect('example.db')` ? SQLite oka file-based database, idi 'example.db' ane file create chesi/open chesi connection istundi.
- Step 2: `connection.cursor()` ? cursor ante oka pointer/handle, deeni tho SQL commands execute chestam.
- Step 3: `cursor.execute(sql)` ? SQL statement (CREATE/INSERT/SELECT/UPDATE) ni SQLite engine ki pampistundi.
- Step 4: Read operations (`SELECT`) ?? `fetchall()`/`fetchone()` v????? ? results Python tuples ga vastay.
- Step 5: Write operations (`INSERT`/`UPDATE`/`DELETE`) tarvatha `connection.commit()` call cheyyali ? lekapothe changes disk lo save avvavu (rollback ayye chance undi).
- Step 6: Pani ayyaka `connection.close()` tho connection close cheyyali ? resources release avutay.


## 1) Connect to an SQLite database

SQLite ante ?? **serverless, file-based** database ? separate server install cheyyakarledu, oka `.db` file matrame chaalu.

In [1]:
# sqlite3 module ni import chestunnam ? idi Python built-in, extra install avasaram ledu
import sqlite3

## Connect to an SQLite database
# 'example.db' ane file create chestundi (already unte open chestundi)
connection = sqlite3.connect('example.db')
connection


## 2) Cursor create cheyyadam

Cursor tho manam SQL commands execute chestam ? idi database tho matlad? object.

In [2]:
# connection nunchi cursor create chestunnam ? commands run cheyyadaniki idi v?????
cursor = connection.cursor()


## 3) Table Create Cheyyadam

`CREATE TABLE IF NOT EXISTS` v??ithe table already unte error radu ? safe way.

In [3]:
## Create a Table
cursor.execute('''
Create Table If Not Exists employees(
            id Integer Primary Key,
            name Text Not Null,
            age Integer,
            department text

            )
''')


## 4) Changes ni Commit Cheyyadam

`INSERT`/`UPDATE`/`DELETE`/`CREATE` tarvatha **commit** cheyyakapothe, changes disk file lo permanent ga save avvavu.

In [4]:
## Commit the changes
connection.commit()


## 5) Data ni Query Cheyyadam (empty table)

Table create chesina ?????? empty ga untundi, so SELECT chesthe rows radu.

In [5]:
cursor.execute('''
Select * from employees

''')


## 6) Data ni Insert Cheyyadam

3 ways lo INSERT statement ? column order ni match cheyyadam chala important. `values` lo icchina order table columns order tho match avvali (id auto ga fill avutundi ? Primary Key).

In [6]:
## insert data in sqlite table
cursor.execute('''
Insert Into employees(name,age,department)
            values('Krish',32,'Data Scientist')

''')

cursor.execute('''
INSERT INTO employees (name, age, department)
VALUES ('Bob', 25, 'Engineering')
''')

cursor.execute('''
INSERT INTO employees (name, age, department)
VALUES ('Charlie', 35, 'Finance')
''')

##
connection.commit()


## 7) Data ni Query Cheyyadam (rows tho)

`fetchall()` ? anni matching rows ni oka list of tuples ga istundi.

In [7]:
## Query the data from the table
cursor.execute('Select * from employees')
rows=cursor.fetchall()

## print the queried data
for row in rows:
    print(row)


(1, 'Krish', 32, 'Data Scientist')
(2, 'Bob', 25, 'Engineering')
(3, 'Charlie', 35, 'Finance')


**Output:**
```
(1, 'Krish', 32, 'Data Scientist')
(2, 'Bob', 25, 'Engineering')
(3, 'Charlie', 35, 'Finance')
```

## 8) Data ni Update Cheyyadam

`WHERE` clause pettakapothe **anni rows** update ayipotay ? chaala careful ga v?????. Idi kuda commit cheyyali.

In [8]:
## Update the data
cursor.execute('''
UPDATE employees
Set age=34
where name="Krish"

''')

connection.commit()


## 9) Verify Update ? malli Query Cheyyadam

In [9]:
## Query the data from the table
cursor.execute('Select * from employees')
rows=cursor.fetchall()

## print the queried data
for row in rows:
    print(row)


(1, 'Krish', 34, 'Data Scientist')
(2, 'Bob', 25, 'Engineering')
(3, 'Charlie', 35, 'Finance')


**Output (Krish age 32 -> 34):**
```
(1, 'Krish', 34, 'Data Scientist')
(2, 'Bob', 25, 'Engineering')
(3, 'Charlie', 35, 'Finance')
```

## 10) Delete Cheyyadam

`WHERE` l?????? table lo unna anni rows delete ay??????? ? production ?? ??????? WHERE ??????? DELETE run che?????.

In [10]:
## Delete a row
cursor.execute('''
DELETE FROM employees
WHERE name = "Bob"
''')

connection.commit()


## 11) Connection Close Cheyyadam

Pani ayyaka connection ni close che???? ? ledante file lock ayi unde chance undi, ithara programs open che?????.

In [11]:
## Close the connection
connection.close()


## 12) SQL Quick Reference (Deep Notes)

| Command | Deeni kosam |
|---|---|
| `CREATE TABLE` | Kotha table nirminchadam (columns + data types + constraints) |
| `INSERT INTO` | Kotha rows add cheyyadam |
| `SELECT` | Data ni query/read cheyyadam |
| `UPDATE` | Unna rows ni modify cheyyadam |
| `DELETE` | Rows ni table nunchi ??????? |
| `WHERE` | Condition ?????? specific rows ni target che??? |
| `PRIMARY KEY` | Prathi row ki unique identifier |
| `NOT NULL` | Aa column ki value ????????? undali |

### Enduku commit() important?
SQLite lo prathi write operation (`INSERT`/`UPDATE`/`DELETE`) oka transaction lo ??????????. `connection.commit()` call che???? ???? aa changes disk file ???? permanent ?? ??????? ? program crash ????? changes ?????? (rollback).

### Best Practices
- Prathi query lo **parameterized queries** (`?` placeholders) v????? ? direct string formatting SQL Injection ki ???? ?????????.
- Pani ayyaka eppudu **connection.close()** cheyyali, leda `with` context manager v?????.
- Bulk inserts ki `cursor.executemany()` v????? performance better.


## 13) WHERE Clause ? Filtering Rows

`WHERE` clause specific rows ni fetch chestundi. Idi `SELECT`, `UPDATE`, `DELETE` anni lo work chestundi.

In [12]:
import sqlite3

# Fresh connection open chestunnam
connection = sqlite3.connect('example.db')
cursor = connection.cursor()

## WHERE tho specific employee fetch cheyyadam
cursor.execute('''
    SELECT * FROM employees
    WHERE department = "Engineering"
''')
for row in cursor.fetchall():
    print(row)


## 14) ORDER BY ? Sort Cheskovadam

`ORDER BY` results ni oka column meeda sort chestundi. `ASC` (default) = small to big, `DESC` = big to small.

In [13]:
## ORDER BY tho employees ni age meeda sort cheyyadam (DESC ? peddha vallaki first)
cursor.execute('''
    SELECT * FROM employees
    ORDER BY age DESC
''')
for row in cursor.fetchall():
    print(row)


(3, 'Charlie', 35, 'Finance')
(1, 'Krish', 34, 'Data Scientist')


## 15) LIKE Operator ? Pattern Matching

`LIKE` tho partial text search cheyyadam. `%` = zero or more characters, `_` = exactly one character.

In [14]:
## 'K' tho start ayye names filter cheyyadam
cursor.execute('''
    SELECT * FROM employees
    WHERE name LIKE "K%"
''')
for row in cursor.fetchall():
    print(row)


(1, 'Krish', 34, 'Data Scientist')


## 16) LIMIT & OFFSET ? Pagination

`LIMIT n` ? max n rows istundi. `OFFSET n` ? first n rows skip chesi remaining istundi. Ee rendu kalipi **pagination** implement cheyyadam easy.

In [15]:
## First 2 employees matrame fetch cheyyadam
cursor.execute('SELECT * FROM employees LIMIT 2')
print('LIMIT 2:')
for row in cursor.fetchall():
    print(row)

## First 1 skip chesi, next 2 fetch cheyyadam
cursor.execute('SELECT * FROM employees LIMIT 2 OFFSET 1')
print('LIMIT 2 OFFSET 1:')
for row in cursor.fetchall():
    print(row)


LIMIT 2:
(1, 'Krish', 34, 'Data Scientist')
(3, 'Charlie', 35, 'Finance')
LIMIT 2 OFFSET 1:
(3, 'Charlie', 35, 'Finance')


## 17) Aggregate Functions ? COUNT, SUM, AVG, MIN, MAX

Aggregate functions anni rows ni combine chesi oka result istay. Data analysis ki chaala important.

In [16]:
## Aggregate functions ? employees table meeda
cursor.execute('SELECT COUNT(*) FROM employees')
print('Total employees:', cursor.fetchone()[0])

cursor.execute('SELECT AVG(age) FROM employees')
print('Average age:', cursor.fetchone()[0])

cursor.execute('SELECT MIN(age), MAX(age) FROM employees')
row = cursor.fetchone()
print('Min age:', row[0], '| Max age:', row[1])


Total employees: 2
Average age: 34.5
Min age: 34 | Max age: 35


## 18) GROUP BY ? Grouping + Aggregation

`GROUP BY` oka column value meeda rows ni group chestundi. Prathi group ki aggregate function apply cheyyadam possible. Example: department wise employee count.

In [17]:
## Department wise employee count
cursor.execute('''
    SELECT department, COUNT(*) as total
    FROM employees
    GROUP BY department
''')
print('Department | Count')
for row in cursor.fetchall():
    print(row)


Department | Count
('Data Scientist', 1)
('Finance', 1)


## 19) fetchone() vs fetchall() vs fetchmany()

| Method | Enti chestundi |
|---|---|
| `fetchone()` | Okke oka row tuple ga istundi, rows ayyipothe None |
| `fetchall()` | Anni matching rows oka list of tuples ga istundi |
| `fetchmany(n)` | Next n rows list ga istundi ? large data ki useful |

In [18]:
cursor.execute('SELECT * FROM employees')

## First row matrame fetch cheyyadam
print('fetchone:', cursor.fetchone())

## Next 1 row fetch cheyyadam (cursor continue avutundi)
print('fetchmany(1):', cursor.fetchmany(1))

## Remaining anni rows fetch cheyyadam
print('fetchall (remaining):', cursor.fetchall())


fetchone: (1, 'Krish', 34, 'Data Scientist')
fetchmany(1): [(3, 'Charlie', 35, 'Finance')]
fetchall (remaining): []


## 20) Parameterized Queries ? SQL Injection Safety

User input ni direct ga SQL string lo pettadam **dangerous** ? SQL Injection attack possible. `?` placeholders use cheyyandi ? sqlite3 automatically safe ga handle chestundi.

In [19]:
## UNSAFE ? string formatting direct ga use cheyyadam (cheyyakandi)
name_input = 'Krish'
# cursor.execute(f"SELECT * FROM employees WHERE name = '{name_input}'")  # DANGEROUS!

## SAFE ? parameterized query tho ? placeholder
cursor.execute(
    'SELECT * FROM employees WHERE name = ?',
    (name_input,)  # tuple ga ivvali ? trailing comma mandatory
)
print('Safe query result:', cursor.fetchone())

## Multiple params example
cursor.execute(
    'SELECT * FROM employees WHERE age > ? AND department = ?',
    (30, 'Data Scientist')
)
print('Age > 30 AND Data Scientist:', cursor.fetchall())


Safe query result: (1, 'Krish', 34, 'Data Scientist')
Age > 30 AND Data Scientist: [(1, 'Krish', 34, 'Data Scientist')]


## 21) executemany() ? Bulk Insert

Multiple rows ni loop lo oka oka insert cheyyadam slow. `executemany()` okesari list of tuples ni insert chestundi ? much faster and cleaner.

In [20]:
## Bulk insert with executemany
new_employees = [
    ('Alice', 28, 'Marketing'),
    ('David', 40, 'Finance'),
    ('Eva',   31, 'Engineering'),
]

# Oka call lo motham list insert avutundi
cursor.executemany(
    'INSERT INTO employees (name, age, department) VALUES (?, ?, ?)',
    new_employees
)
connection.commit()

## Verify cheyyadam
cursor.execute('SELECT * FROM employees')
for row in cursor.fetchall():
    print(row)


(1, 'Krish', 34, 'Data Scientist')
(3, 'Charlie', 35, 'Finance')
(4, 'Alice', 28, 'Marketing')
(5, 'David', 40, 'Finance')
(6, 'Eva', 31, 'Engineering')


## 22) Context Manager (`with`) ? Auto Commit / Rollback

`with connection:` block v??ite:
- Block success ga finish aite ? **auto commit**
- Block lo exception vasthe ? **auto rollback**

Manual `commit()` / `rollback()` calls cheyyanakkarledu ? clean and safe pattern.

In [21]:
## with block ? auto commit
with connection:
    connection.execute(
        'INSERT INTO employees (name, age, department) VALUES (?, ?, ?)',
        ('Frank', 27, 'HR')
    )
    # ikkade exception leka ante auto commit avutundi

## Rollback demo ? with block lo exception raise cheyyadam
try:
    with connection:
        connection.execute(
            'INSERT INTO employees (name, age, department) VALUES (?, ?, ?)',
            ('Ghost', 99, 'NoWhere')
        )
        raise ValueError('Idi intentional error ? rollback test')  # force rollback
except ValueError as e:
    print('Rolled back because:', e)

cursor.execute('SELECT * FROM employees ORDER BY id')
print('After with-block demo:')
for row in cursor.fetchall():
    print(row)


Rolled back because: Idi intentional error ? rollback test
After with-block demo:
(1, 'Krish', 34, 'Data Scientist')
(3, 'Charlie', 35, 'Finance')
(4, 'Alice', 28, 'Marketing')
(5, 'David', 40, 'Finance')
(6, 'Eva', 31, 'Engineering')
(7, 'Frank', 27, 'HR')


## 23) sqlite3.Row ? Dict-style Row Access

Default ga rows tuples ga vastay ? index tho access cheyyali (`row[0]`). `connection.row_factory = sqlite3.Row` set chesthe **column name tho access** cheyyadam possible: `row['name']`.

In [22]:
## row_factory set cheyyadam ? once set chesthe anni future cursors ki apply avutundi
connection.row_factory = sqlite3.Row
cursor2 = connection.cursor()  # new cursor ? row_factory tho

cursor2.execute('SELECT * FROM employees LIMIT 2')
for row in cursor2.fetchall():
    # Column name tho access ? idi tuple index kante readable
    print(f"Name: {row['name']} | Age: {row['age']} | Dept: {row['department']}")

# Dict ga convert cheyyadam
cursor2.execute('SELECT * FROM employees LIMIT 1')
row = cursor2.fetchone()
print('Dict:', dict(row))


Name: Krish | Age: 34 | Dept: Data Scientist
Name: Charlie | Age: 35 | Dept: Finance
Dict: {'id': 1, 'name': 'Krish', 'age': 34, 'department': 'Data Scientist'}


## 24) In-Memory Database ? Testing ki Perfect

`':memory:'` ane special string use chesthe database disk ki radu ? RAM lo matrame untundi. Program close aite data pothundi. Unit tests, quick experiments ki perfect ? speed chaala fast.

In [23]:
## In-memory database ? disk file create kadu
mem_con = sqlite3.connect(':memory:')
mem_con.row_factory = sqlite3.Row
mem_cur = mem_con.cursor()

mem_cur.execute('''CREATE TABLE temp_test (
    id INTEGER PRIMARY KEY,
    value TEXT
)''')

mem_cur.executemany('INSERT INTO temp_test (value) VALUES (?)',
                    [('apple',), ('banana',), ('cherry',)])
mem_con.commit()

mem_cur.execute('SELECT * FROM temp_test')
print('In-memory rows:')
for row in mem_cur.fetchall():
    print(dict(row))

mem_con.close()  # close aite data disappear avutundi
print('mem_con closed ? data gone')


In-memory rows:
{'id': 1, 'value': 'apple'}
{'id': 2, 'value': 'banana'}
{'id': 3, 'value': 'cherry'}
mem_con closed ? data gone


## 25) Introspection ? Table Structure Chudadam

`PRAGMA table_info(tablename)` ? table lo unna columns, types, constraints anni chupistundi. `sqlite_master` nunchi table list chudochu.

In [24]:
regular_con = sqlite3.connect('example.db')

## PRAGMA tho table column details
print('--- employees table_info ---')
for row in regular_con.execute('PRAGMA table_info(employees)'):
    print(row)  # (cid, name, type, notnull, default_value, pk)

## DB lo unna tables list
print('\n--- All tables ---')
for row in regular_con.execute("SELECT name FROM sqlite_master WHERE type='table'"):
    print(row)

regular_con.close()


--- employees table_info ---
(0, 'id', 'INTEGER', 0, None, 1)
(1, 'name', 'TEXT', 1, None, 0)
(2, 'age', 'INTEGER', 0, None, 0)
(3, 'department', 'TEXT', 0, None, 0)

--- All tables ---
('employees',)


## 27) Real-World Example ? Sales Database

Ee section lo oka real-world **sales** table create chesi, bulk data insert chesi, business queries run chestam. Idi common interview + job scenario.

In [25]:
import sqlite3

# Fresh connection ? sales.db file create avutundi
conn = sqlite3.connect('sales.db')
conn.row_factory = sqlite3.Row  # column name access kosam
cur = conn.cursor()

## sales table create cheyyadam
cur.execute('''
    CREATE TABLE IF NOT EXISTS sales (
        id      INTEGER PRIMARY KEY AUTOINCREMENT,
        date    TEXT    NOT NULL,
        product TEXT    NOT NULL,
        sales   INTEGER NOT NULL,
        region  TEXT    NOT NULL
    )
''')
conn.commit()
print('sales table created')


sales table created


## 28) Bulk Insert ? executemany() tho Sales Data

Real data ante many rows untundi. `executemany()` tho oka list of tuples ni okesari insert cheyyadam ? loop lo oka oka insert cheyadam kante **much faster**.

In [26]:
## Sales data ? list of tuples
sales_data = [
    ('2023-01-01', 'Product1', 100, 'North'),
    ('2023-01-02', 'Product2', 200, 'South'),
    ('2023-01-03', 'Product1', 150, 'East'),
    ('2023-01-04', 'Product3', 250, 'West'),
    ('2023-01-05', 'Product2', 300, 'North'),
    ('2023-01-06', 'Product3', 180, 'South'),
    ('2023-01-07', 'Product1', 220, 'West'),
    ('2023-01-08', 'Product2', 170, 'East'),
    ('2023-01-09', 'Product3', 310, 'North'),
    ('2023-01-10', 'Product1', 130, 'South'),
]

## Motham list okesari insert cheyyadam ? executemany
cur.executemany('''
    Insert into sales(date,product,sales,region)
                values(?,?,?,?)
''', sales_data)

conn.commit()
print(f'{cur.rowcount} rows inserted')


10 rows inserted


## 29) All Sales Data Query Cheyyadam

In [27]:
## Anni rows fetch chesi print cheyyadam
cur.execute('SELECT * FROM sales')
print(f"{'ID':<4} {'Date':<12} {'Product':<10} {'Sales':<7} Region")
print('-' * 45)
for row in cur.fetchall():
    print(f"{row['id']:<4} {row['date']:<12} {row['product']:<10} {row['sales']:<7} {row['region']}")


ID   Date         Product    Sales   Region
---------------------------------------------
1    2023-01-01   Product1   100     North
2    2023-01-02   Product2   200     South
3    2023-01-03   Product1   150     East
4    2023-01-04   Product3   250     West
5    2023-01-05   Product2   300     North
6    2023-01-06   Product3   180     South
7    2023-01-07   Product1   220     West
8    2023-01-08   Product2   170     East
9    2023-01-09   Product3   310     North
10   2023-01-10   Product1   130     South


## 30) Region wise Total Sales ? GROUP BY

Prathi region lo total ela undo chudadam ? business lo most used query pattern.

In [28]:
## Region wise total sales ? GROUP BY + SUM
cur.execute('''
    SELECT region,
           SUM(sales)  AS total_sales,
           COUNT(*)    AS num_transactions,
           ROUND(AVG(sales), 2) AS avg_sales
    FROM sales
    GROUP BY region
    ORDER BY total_sales DESC
''')
print(f"{'Region':<10} {'Total':>8} {'Txns':>6} {'Avg':>8}")
print('-' * 36)
for row in cur.fetchall():
    print(f"{row['region']:<10} {row['total_sales']:>8} {row['num_transactions']:>6} {row['avg_sales']:>8}")


Region        Total   Txns      Avg
------------------------------------
North           710      3   236.67
South           510      3    170.0
West            470      2    235.0
East            320      2    160.0


## 31) Product wise Sales ? Best Seller Konkovadam

`ORDER BY total_sales DESC` + `LIMIT 1` tho top selling product kanukodadam.

In [29]:
## Product wise total ? best seller top lo
cur.execute('''
    SELECT product,
           SUM(sales) AS total_sales
    FROM sales
    GROUP BY product
    ORDER BY total_sales DESC
''')
print(f"{'Product':<12} {'Total Sales':>12}")
print('-' * 26)
for row in cur.fetchall():
    print(f"{row['product']:<12} {row['total_sales']:>12}")

## Best seller oka line lo
cur.execute('SELECT product, SUM(sales) AS total FROM sales GROUP BY product ORDER BY total DESC LIMIT 1')
best = cur.fetchone()
print(f"\nBest Seller: {best['product']} ({best['total']} units)")


Product       Total Sales
--------------------------
Product3              740
Product2              670
Product1              600

Best Seller: Product3 (740 units)


## 32) Date Range Filter ? BETWEEN

Specific date range lo sales chudadam ? `WHERE date BETWEEN 'start' AND 'end'` v??ithe easy.

In [30]:
## Jan 1 to Jan 5 sales matrame ? date range filter
cur.execute('''
    SELECT date, product, sales, region
    FROM sales
    WHERE date BETWEEN '2023-01-01' AND '2023-01-05'
    ORDER BY date
''')
print('Sales from 2023-01-01 to 2023-01-05:')
for row in cur.fetchall():
    print(dict(row))


Sales from 2023-01-01 to 2023-01-05:
{'date': '2023-01-01', 'product': 'Product1', 'sales': 100, 'region': 'North'}
{'date': '2023-01-02', 'product': 'Product2', 'sales': 200, 'region': 'South'}
{'date': '2023-01-03', 'product': 'Product1', 'sales': 150, 'region': 'East'}
{'date': '2023-01-04', 'product': 'Product3', 'sales': 250, 'region': 'West'}
{'date': '2023-01-05', 'product': 'Product2', 'sales': 300, 'region': 'North'}


## 33) Subquery ? Average Kante Ekkuva Sales

Average sales kante ekkuva unna rows filter cheyyadam ? subquery tho dynamically average calculate avutundi.

In [31]:
## Average kante ekkuva sales unna rows
cur.execute('''
    SELECT date, product, sales, region
    FROM sales
    WHERE sales > (SELECT AVG(sales) FROM sales)
    ORDER BY sales DESC
''')
avg_val = conn.execute('SELECT ROUND(AVG(sales),2) FROM sales').fetchone()[0]
print(f'Overall average sales: {avg_val}')
print('Rows above average:')
for row in cur.fetchall():
    print(dict(row))


Overall average sales: 201.0
Rows above average:
{'date': '2023-01-09', 'product': 'Product3', 'sales': 310, 'region': 'North'}
{'date': '2023-01-05', 'product': 'Product2', 'sales': 300, 'region': 'North'}
{'date': '2023-01-04', 'product': 'Product3', 'sales': 250, 'region': 'West'}
{'date': '2023-01-07', 'product': 'Product1', 'sales': 220, 'region': 'West'}


## 34) Cleanup ? Close Connection & Remove Demo Files

In [32]:
import os

# Demo db files tisi remove cheyyadam — already removed / locked aite gracefully handle
for db_file in ('example.db', 'sales.db', 'close_demo.db'):
    try:
        if os.path.exists(db_file):
            os.remove(db_file)
            print(f'{db_file} removed')
        else:
            print(f'{db_file} already gone')
    except OSError as e:
        print(f'Could not remove {db_file}: {e}')


Could not remove example.db: [WinError 32] The process cannot access the file because it is being used by another process: 'example.db'
Could not remove sales.db: [WinError 32] The process cannot access the file because it is being used by another process: 'sales.db'
close_demo.db already gone


## 35) connection.close() ? Deep Dive

`connection.close()` ante just oka line ? kaani idi cheyyadam **mandatory**. Cheyyadam marchipote:

- DB file **lock** ayyi untundi ? ithara programs open cheyyalerru
- Uncommitted changes **lost** avutayi
- Memory **leak** ? connection object RAM lo undi untundi
- Multiple connections open aite **data corruption** risk undi

In [33]:
import sqlite3, os

# Connection open cheyyadam
connection = sqlite3.connect('close_demo.db')
cursor = connection.cursor()

cursor.execute('CREATE TABLE IF NOT EXISTS demo (id INTEGER PRIMARY KEY, val TEXT)')
cursor.execute("INSERT INTO demo (val) VALUES ('hello')")
connection.commit()

## connection.close() cheyyadam ? resources release avutay
connection.close()
print('Connection closed successfully')

## Close chesina tarvata execute cheyyadam try chesthe ? ProgrammingError vastuundi
try:
    cursor.execute('SELECT * FROM demo')  # closed connection meeda query
except Exception as e:
    print(f'Error after close: {type(e).__name__}: {e}')


Connection closed successfully
Error after close: ProgrammingError: Cannot operate on a closed database.


### 35.1) in_transaction ? Commit Chesaka Close Cheyyadam

`connection.in_transaction` ? uncommitted changes unnayi ante `True` istundi. Close cheyyadaniki mundu idi check cheyyadam good practice.

In [34]:
connection = sqlite3.connect('close_demo.db')
cursor = connection.cursor()

## Commit cheyyakunda insert cheyyadam
cursor.execute("INSERT INTO demo (val) VALUES ('world')")

## in_transaction check ? True ante uncommitted changes unnay
print('in_transaction (before commit):', connection.in_transaction)

connection.commit()
print('in_transaction (after commit):', connection.in_transaction)

connection.close()
print('Closed cleanly')


in_transaction (before commit): True
in_transaction (after commit): False
Closed cleanly


### 35.2) try / finally ? Exception Vachina Kuda Close Guarantee

Code lo exception vasthe normal `connection.close()` skip avutundi. `finally` block **always** run avutundi ? exception vassinaa, vassinaa. Idi production code lo safest pattern.

In [35]:
connection = sqlite3.connect('close_demo.db')
cursor = connection.cursor()

try:
    cursor.execute('SELECT * FROM demo')
    rows = cursor.fetchall()
    print('Rows fetched:', rows)
    raise RuntimeError('Simulated crash mid-operation')  # force error
except RuntimeError as e:
    print(f'Caught error: {e}')
finally:
    # Exception vassinaa, vassinaa ? idi always run avutundi
    connection.close()
    print('Connection closed in finally block')


Rows fetched: [(1, 'hello'), (2, 'world')]
Caught error: Simulated crash mid-operation


Connection closed in finally block


### 35.3) `with` Context Manager ? Best Practice (Recommended)

`with sqlite3.connect(...) as conn:` v??ithe:

- Block success: **auto commit**
- Block lo exception: **auto rollback**
- **BUT** ? `with` block SQLite lo auto close cheyyadu! `close()` still manual ga cheyyali

> ?? **Tip:** `with` + `finally` combine cheyyandi ? commit/rollback AND close guarantee.

In [36]:
## with block ? auto commit/rollback, manual close
connection = sqlite3.connect('close_demo.db')

try:
    with connection:  # with block ? exception aite rollback, success aite commit
        connection.execute("INSERT INTO demo (val) VALUES ('with_pattern')")
        # ikkade exception leka ? auto commit
    print('Commit done via with block')
finally:
    connection.close()  # with block close cheyyadu ? manually cheyyali
    print('Connection closed')

## Verify ? close chesina tarvata new connection open chesi read cheyyadam
verify_conn = sqlite3.connect('close_demo.db')
rows = verify_conn.execute('SELECT * FROM demo').fetchall()
print('Verified rows:', rows)
verify_conn.close()


Commit done via with block
Connection closed
Verified rows: [(1, 'hello'), (2, 'world'), (3, 'with_pattern')]


### 35.4) Multiple Connections ? Close Order Matters

Same `.db` file meeda multiple connections open aite ? oka connection write chestundaga inkoka wait chestundi (SQLite default). Anni connections close cheyyakunda program exit aite file lock stuck avutundi.

In [37]:
## Multiple connections same db ki
conn1 = sqlite3.connect('close_demo.db')
conn2 = sqlite3.connect('close_demo.db')  # same file, rendu connections

conn1.execute("INSERT INTO demo (val) VALUES ('conn1_write')")
conn1.commit()

## conn2 recent data read cheyyagaladu (oka different connection)
rows = conn2.execute('SELECT val FROM demo ORDER BY id').fetchall()
print('conn2 reads:', [r[0] for r in rows])

## Rendu connections also close cheyyali
conn1.close()
conn2.close()
print('Both connections closed')


conn2 reads: ['hello', 'world', 'with_pattern', 'conn1_write']
Both connections closed


### 35.5) Cleanup ? Demo DB Remove Cheyyadam

In [38]:
import os

# Demo db files tisi remove cheyyadam — already removed / locked aite gracefully handle
for db_file in ('example.db', 'sales.db', 'close_demo.db'):
    try:
        if os.path.exists(db_file):
            os.remove(db_file)
            print(f'{db_file} removed')
        else:
            print(f'{db_file} already gone')
    except OSError as e:
        print(f'Could not remove {db_file}: {e}')


Could not remove example.db: [WinError 32] The process cannot access the file because it is being used by another process: 'example.db'
Could not remove sales.db: [WinError 32] The process cannot access the file because it is being used by another process: 'sales.db'
close_demo.db removed


## 26) Cleanup ? Close Connection & Remove Demo DB

Notebook rerun lo duplicate data raakapovadam kosam demo `.db` file cleanup chestunnam.

In [39]:
import os

# Demo db files tisi remove cheyyadam — already removed / locked aite gracefully handle
for db_file in ('example.db', 'sales.db', 'close_demo.db'):
    try:
        if os.path.exists(db_file):
            os.remove(db_file)
            print(f'{db_file} removed')
        else:
            print(f'{db_file} already gone')
    except OSError as e:
        print(f'Could not remove {db_file}: {e}')


Could not remove example.db: [WinError 32] The process cannot access the file because it is being used by another process: 'example.db'
Could not remove sales.db: [WinError 32] The process cannot access the file because it is being used by another process: 'sales.db'
close_demo.db already gone
